# this notebook is called by launcher.ipynb. it cannot run on it's own

In [ ]:
# this assumes that:
#    - dates is defined when as follows: dates = {"start_date": "2019-01-01", "end_date": "2019-02-01"}
#    - year0 is defined
#    - do_litetracks, do_planned_routes, and do_airspace_assignments are defined and set

In [ ]:
# do_litetracks = True
# do_flightplans = False
# do_airspace_assignments = False

In [ ]:
print("starting: ", dates)

In [ ]:
airports = [
    "KADW", "KATL", "KBOS", "KBWI", "KCLT", "KDCA", "KDEN", "KDFW", "KDTW",
    "KEWR", "KFLL", "KIAD", "KIAH", "KJFK", "KLAS", "KLAX", "KLGA", "KMCO",
    "KMDW", "KMEM", "KMIA", "KMSP", "KORD", "KPHL", "KPHX", "KSAN", "KSDF",
    "KSEA", "KSFO", "KSLC", "KTPA", "PANC", "PHNL",
]

# UDFs

In [ ]:
#Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string

@F.udf("string")
def to_date(ts):
    return datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

## litetrack stuff

In [ ]:
points_schema = T.StructType([
    T.StructField("points", T.ArrayType(
        T.StructType([
            T.StructField("primary_key", T.StringType(), True),
            T.StructField("time", T.LongType(), True),
            T.StructField("latitude", T.DoubleType(), True),
            T.StructField("longitude", T.DoubleType(), True),
            T.StructField("altitude", T.FloatType(), True),
            T.StructField("course", T.FloatType(), True),
            T.StructField("speed", T.FloatType(), True),
            T.StructField("source_point_keys", T.ArrayType(
                T.StructType([
                    T.StructField("element", T.StringType(), True),
                ]), True), True),
            T.StructField("acceleration", T.DoubleType(),True),
            T.StructField("type", T.StringType(), True),
            T.StructField("airspace_key", T.StringType(), True),
            T.StructField("tas", T.FloatType(), True),
            T.StructField("ias", T.FloatType(), True),
            T.StructField("along_track_distance", T.FloatType(), True),
            T.StructField("derived_point_key", T.StringType(), True),
        ]), True), True)
    ])

In [ ]:
# a function that will convert the points array into a string containing n-tuples of data for each litetrack point

none_string = "*"
separator_string = ":"

def stringify_litetrack_points(array_of_points:T.ArrayType(points_schema)) -> str:
    output = ""
    
    for x in array_of_points:
        output += (none_string if x.time is None else str(x.time)) + " "
        output += (none_string if x.latitude is None else f'{x.latitude:.6f}') + " "
        output += (none_string if x.longitude is None else f'{x.longitude:.6f}') + " "
        output += (none_string if x.altitude is None else f'{x.altitude:.0f}') + " "
        output += (none_string if x.course is None else f'{x.course:.0f}') + " "
        output += (none_string if x.speed is None else f'{x.speed:.0f}') + separator_string

    # drop the trailing tuple separator
    if len(array_of_points) > 0:
        output = output[0:len(separator_string) * -1]

    return output

stringify_litetrack_points_udf = F.udf(stringify_litetrack_points, T.StringType())

## airspaceassignment stuff

In [ ]:
aa_point_json =  '{"fields":[{"metadata":{},"name":"entry_point","nullable":true,"type":{"fields":[{"metadata":{},"name":"primary_key","nullable":true,"type":"string"},{"metadata":{},"name":"time","nullable":true,"type":"long"},{"metadata":{},"name":"latitude","nullable":true,"type":"double"},{"metadata":{},"name":"longitude","nullable":true,"type":"double"},{"metadata":{},"name":"altitude","nullable":true,"type":"float"},{"metadata":{},"name":"course","nullable":true,"type":"float"},{"metadata":{},"name":"speed","nullable":true,"type":"float"},{"metadata":{},"name":"source_point_keys","nullable":true,"type":{"containsNull":true,"elementType":"string","type":"array"}},{"metadata":{},"name":"airspace_key","nullable":true,"type":"string"},{"metadata":{},"name":"distance","nullable":true,"type":"double"},{"metadata":{},"name":"climb_rate","nullable":true,"type":"double"}],"type":"struct"}}],"type":"struct"}'
aa_point_schema = T.StructType.fromJson(json.loads(aa_point_json))

In [ ]:
# functions to stringify the entry/exit point structure

def strinigfy_airspaceassignment_point(aa_point:aa_point_schema) -> str:
    output = ""
    
    # output += str(aa_point.time) + " "
    # output += f'{aa_point.latitude:.6f}' + " "
    # output += f'{aa_point.longitude:.6f}' + " "
    # output += f'{aa_point.altitude:.0f}'
    
    return output

strinigfy_airspaceassignment_point_udf = F.udf(strinigfy_airspaceassignment_point, T.StringType())

In [ ]:
# functions to stringify the airspace assignment row

none_string = "*"

def strinigfy_airspaceassignment(artcc:T.StringType, airspace_identifier:T.StringType, airspace_class:T.StringType, airspace_subclass:T.StringType, entry_point:aa_point_schema, exit_point:aa_point_schema, duration:T.DoubleType, distance:T.DoubleType) -> str:
    output = ""
    
    # output += artcc + " "
    # output += "'" + airspace_identifier + "' "
    # # output += airspace_subclass + " "
    # output += (none_string if airspace_subclass is None else airspace_subclass) + " "
    # output += strinigfy_airspaceassignment_point(entry_point) + " "
    # output += strinigfy_airspaceassignment_point(exit_point) + " "
    # output += f'{duration:.1f}' + " "
    # output += f'{distance:.1f}'
    
    return output

strinigfy_airspaceassignment_udf = F.udf(strinigfy_airspaceassignment, T.StringType())

## Make the track_key master list

In [ ]:
print("building track_key master list", datetime.now())

In [ ]:
print("   getting df_flightplanseries_track_keys", datetime.now())
df_FlightplanSeries = (
    api.dataframe("FlightplanSeries", **dates, metadata=True)
    .select(
        "track_key",
        F.col("initial_departure_aerodrome").alias("orig"),
        F.col("final_destination_aerodrome").alias("dest"),
    )
    .filter(F.col("orig") != F.col("dest"))
    .filter(F.col("orig").isin(airports) | F.col("dest").isin(airports))
    .drop("orig", "dest")
)

print("   getting df_plannedroutes_track_keys", datetime.now())
df_PlannedRoute = (
    api.dataframe("PlannedRoute", **dates, partition_filters=api.custom_partitions("ASSOCIATED"), metadata=True)
    .select(
        "track_key",
        F.col("labels.is_lfpd").alias("is_lfpd"),
        F.col("departure_aerodrome").alias("orig"),
        F.col("destination_aerodrome").alias("dest"),
    )
    .filter(F.col("is_lfpd") == 'true')
    .filter(F.col("orig") != F.col("dest"))
    .filter(F.col("orig").isin(airports) | F.col("dest").isin(airports))
    .drop("is_lfpd", "orig", "dest")
)

print("   getting df_airspaceassignments_track_keys", datetime.now())
df_AirspaceAssignment = (
    api.dataframe("AirspaceAssignment", **dates, partition_filters=[
        api.custom_partitions(["ASSOCIATED", "StaticEramAirspace", "SECTOR"]),
        api.custom_partitions(["ASSOCIATED", "StaticEramAirspace", "TRACON"]),
        api.custom_partitions(["ASSOCIATED", "FlightInformationRegion"]),
    ], metadata=True)
    .select(
        "track_key",
    )
)

print("   getting df_litetracks_track_keys", datetime.now())
df_LiteTrack = (
    api.dataframe("LiteTrack", **dates, partition_filters=api.custom_partitions("ASSOCIATED"))
    .select(
        "track_key",
    )
)

# print("   df_FlightplanSeries size:", df_FlightplanSeries.count(), datetime.now())
# print("   df_PlannedRoute size:", df_PlannedRoute.count(), datetime.now())
# print("   df_AirspaceAssignment size:", df_AirspaceAssignment.count(), datetime.now())
# print("   df_LiteTrack size:", df_LiteTrack.count(), datetime.now())

df_track_key_master = df_FlightplanSeries.join(df_PlannedRoute, on=["track_key"], how="inner")
df_track_key_master = df_track_key_master.join(df_AirspaceAssignment, on=["track_key"], how="inner")
df_track_key_master = df_track_key_master.join(df_LiteTrack, on=["track_key"], how="inner")
df_track_key_master = df_track_key_master.select("track_key").distinct()

# df_track_key_master.printSchema()

print("   df_track_key_master size:", df_track_key_master.count(), datetime.now())
df_track_key_master.cache()

# (
#         df_track_key_master
#         .write.option("header", True)
#         .csv("CRAFT/" + year0 + "/nas_year_full/track_key_master", compression="None", mode="append")
# )


# Load LiteTrack

In [ ]:
# get litetracks for the date range of interest

#if do_litetracks:
    
print("   retrieving litetracks: ", datetime.now())

df_litetracks = (
    api.dataframe("LiteTrack", **dates, partition_filters=api.custom_partitions("ASSOCIATED"))
    .select(
        "track_key",
        F.col("start_time").alias("start_epoch_millisec"),
        F.col("end_time").alias("end_epoch_millisec"),
        "points",
    )
    .join(df_track_key_master, on=["track_key"], how="inner")
    .withColumn("trackpoints", stringify_litetrack_points_udf("points"))
    .drop("points")
)

# Load PlannedRoute (with route expansion)

In [ ]:
#if do_planned_routes:

print("   retrieving planned_routes: ", datetime.now())

## get fix_key and associated icao_region
df_arincfix = (
    api.dataframe("ArincFix", **dates, metadata=True)
    .select(
        F.col("primary_key").alias("fix_key"),
        F.col("identification.icao_region").alias("icao_region"),
    )
)

## get the route expansion and join with arincfix to get the icao_region for each fix
df_plannedroutes_raw = (
    api.dataframe("PlannedRoute", **dates, partition_filters=api.custom_partitions("ASSOCIATED"), metadata=True)
    .select(
        "track_key",
        "route",
        F.col("total_route_length").alias("route_length_nm"),
        F.col("labels.is_lfpd").alias("is_lfpd"),
        F.explode("legs").alias("leg"),
        F.col("leg.sequence_number").alias("seqnum"),
        F.col("leg.path_terminator.fix_name").alias("fix_name"),
        F.col("leg.path_terminator.fix_key").alias("fix_key"),
    )
    .join(df_track_key_master, on=["track_key"], how="inner")
    .filter(F.col("is_lfpd") == 'true')
    .withColumn("route_length_nm", F.round("route_length_nm", 1))
    .join(df_arincfix, on=["fix_key"], how="inner")
    .withColumn("fix_name_uniq", F.concat("fix_name", F.lit("("), "icao_region", F.lit(")")))
    .drop("legs", "leg", "is_lfpd", "icao_region", "fix_key")
)

## remove any duplicate fixes in the route expansion
window = Window.partitionBy("track_key", "fix_name").orderBy(col("seqnum").asc())
df_plannedroute_raw_dedup = (
    df_plannedroutes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
    .orderBy("track_key", "seqnum")
)

## make plannedroutes dataframe with route expansion strings fix_name_expansion and fix_name_uniq_expansion 
df_plannedroutes = (
    df_plannedroute_raw_dedup
        .groupby("track_key", "route_length_nm", "route")
        .agg(F.sort_array(F.collect_list(F.struct("seqnum", "fix_name"))).alias("fix_name_list"), F.sort_array(F.collect_list(F.struct("seqnum", "fix_name_uniq"))).alias("fix_name_uniq_list"))
        .withColumn("fix_name_expansion", F.concat_ws(".", F.col("fix_name_list.fix_name")))
        .withColumn("fix_name_uniq_expansion", F.concat_ws(".", F.col("fix_name_uniq_list.fix_name_uniq")))
        .drop("fix_name_list", "fix_name_uniq_list")
)

# Load FlightplanSeries

In [ ]:
## get the flightplans for tracks to and from airports of interest

#if do_flightplans:

print("   retrieving flightplans: ", datetime.now())

df_flightplanseries = (
    api.dataframe("FlightplanSeries", **dates, metadata=True)
    .select(
        "track_key",
        to_date("metadata.effective_start_date").alias("start_date"),
        to_date("metadata.effective_end_date").alias("end_date"),
        "callsign",
        "aircraft_type",
        "mode_s_code",
        F.col("initial_departure_aerodrome").alias("orig"),
        F.col("final_destination_aerodrome").alias("dest"),
    )
    .join(df_track_key_master, on=["track_key"], how="inner")
    .withColumn("month", F.substring("start_date", 5, 2))
#    .withColumn("carrier", extract_carrier_udf("callsign"))
    .withColumn("carrier", F.regexp_extract("callsign", "^([A-Za-z]+)\d*[A-Za-z]*$", 1))
    .filter(F.col("orig") != F.col("dest"))
    .filter(F.col("orig").isin(airports) | F.col("dest").isin(airports))
)

# Load AirspaceAssignments

In [ ]:
# if do_airspace_assignments:

print("   retrieving airspace_assignments: ", datetime.now())

## get the raw AA for the date range
df_airspaceassignments_raw = (
    api.dataframe("AirspaceAssignment", **dates, partition_filters=[
        api.custom_partitions(["ASSOCIATED", "StaticEramAirspace", "SECTOR"]),
        api.custom_partitions(["ASSOCIATED", "StaticEramAirspace", "TRACON"]),
        api.custom_partitions(["ASSOCIATED", "FlightInformationRegion"]),
    ], metadata=True)
    .join(df_track_key_master, on=["track_key"], how="inner")
)

## define the point schema
df_aa_point = (df_airspaceassignments_raw.select("entry_point"))
aa_point_json = df_aa_point.schema.json()   
aa_point_schema = T.StructType.fromJson(json.loads(aa_point_json))


## sort the df_aa_raw data by track_key and entry_point_time,
## create an "artcc" column from the meta_custom_partitions[1],
## and stringify the needed data to an "airspace_assignment" column
df_airspaceassignments_stringified = (
    df_airspaceassignments_raw
    .sort(F.col("track_key"), F.col("entry_point.time"))
    .withColumn("artcc", df_airspaceassignments_raw.metadata.custom_partitions[1])
    .withColumn("airspace_assignment", strinigfy_airspaceassignment_udf("artcc", "airspace_identifier", "airspace_class", "airspace_subclass", "entry_point", "exit_point", "known_visit_duration", "known_distance_covered"))
    .select(
        "track_key",
        "airspace_assignment",
    )
)

## group by track_key and concatenate grouped "airspace_assignment"
df_airspaceassignments = (
    df_airspaceassignments_stringified
   .groupBy("track_key")
   .agg(F.concat_ws(":", F.collect_list("airspace_assignment")).alias("airspace_assignments"))
)

In [ ]:
#stop

# Join the dataframes

In [ ]:
print("   joining df_litetracks to df_flightplanseries: ", datetime.now())

In [ ]:
df_temp0 = df_litetracks.join(df_flightplanseries, on=["track_key"], how="inner")

In [ ]:
print("   joining in df_plannedroutes: ", datetime.now())

In [ ]:
df_temp1 = df_temp0.join(df_plannedroutes, on=["track_key"], how="inner")
#df_temp1 = df_temp0.join(df_plannedroute_raw_dedup, on=["track_key"], how="inner")

In [ ]:
print("   joining df_airspaceassignments: ", datetime.now())

In [ ]:
df_joined = df_temp1.join(df_airspaceassignments, on=["track_key"], how="inner")
#df_joined = df_temp1.join(df_airspaceassignments_stringified, on=["track_key"], how="inner")

# Create arrivals and departures dataframes

In [ ]:
df_arrivals = (
    df_joined.withColumn("airport", F.col("dest"))
    .withColumn("operation", F.lit("ARRIVALS"))
    .filter(F.col("airport").isin(airports))
)

In [ ]:
df_departures = (
    df_joined.withColumn("airport", F.col("orig"))
    .withColumn("operation", F.lit("DEPARTURES"))
    .filter(F.col("airport").isin(airports))
)

# Combine arrival and departure into a master dataframe

In [ ]:
print("   creating master dataframe: ", datetime.now())

In [ ]:
df_master = df_arrivals.unionByName(df_departures)

In [ ]:
df_master.printSchema()

In [ ]:
df_master.cache()

In [ ]:
# print("   starting saving df_master: ", datetime.now())

# (
#     df_master
#     .write.option("header", True).partitionBy(["airport", "operation", "month"])
#     .csv("CRAFT/" + year0 + "/nas_year_full/df_master", compression="gzip", mode="overwrite")
# )

# print("   completed saving df_master: ", datetime.now())

In [ ]:
#stop

# Save LiteTracks

In [ ]:
if do_litetracks:
    print("   started saving litetracks: ", datetime.now())

    df_litetracks_output = (
        df_master
#        .withColumn("trackpoints", stringify_litetrack_points_udf("points"))
        .select(
            "track_key",
            "start_date",
            "end_date",
            "orig",
            "dest",
            "mode_s_code",
            "callsign",
            "carrier",
            "aircraft_type",
            "trackpoints",
            "airport",
            "operation",
            "month",
        )
    )

    (
        df_litetracks_output
#        .repartition("start_date", "dest", "airport", "operation")
#        .repartition("track_key", "dest", "airport", "operation")
#        .repartition("airport", "operation", "month")
#        .repartition(200)
        .write.option("header", True).partitionBy(["airport", "operation", "month"])
        .csv("CRAFT/" + year0 + "/nas_year_full/litetracks", compression="gzip", mode="append")
    )

    print("   completed saving litetracks: ", datetime.now()) 

# Save FlightPlanSeries Data

In [ ]:
if do_flightplans:
    print("   started saving flightplans: ", datetime.now())

    df_flightplanseries_output = (
        df_master
#        .groupby("track_key", "route_length_nm", "route")
#        .agg(F.sort_array(F.collect_list(F.struct("seqnum", "fix_name"))).alias("fix_name_list"), F.sort_array(F.collect_list(F.struct("seqnum", "fix_name_uniq"))).alias("fix_name_uniq_list"))
#        .withColumn("fix_name_expansion", F.concat_ws(".", F.col("fix_name_list.fix_name")))
#        .withColumn("fix_name_uniq_expansion", F.concat_ws(".", F.col("fix_name_uniq_list.fix_name_uniq")))
        .select(
            "track_key",
            "start_date",
            "end_date",
            "start_epoch_millisec",
            "end_epoch_millisec",
            "orig",
            "dest",
            "callsign",
            "carrier",
            "mode_s_code",
            "aircraft_type",
            "route",
            "fix_name_expansion",
            "fix_name_uniq_expansion",
            "route_length_nm",
            "airport",
            "operation",
            "month",
        )
    )

    (
        df_flightplanseries_output
#        .repartition("start_date", "dest", "airport", "operation")
#        .repartition("track_key", "dest", "airport", "operation")
#        .repartition("airport", "operation", "month")
#       .repartition(200)
        .write.option("header", True).partitionBy(["airport", "operation", "month"])
        .csv("CRAFT/" + year0 + "/nas_year_full/flightplans", compression="gzip", mode="append")
    )

    print("   completed saving flightplans: ", datetime.now())

# Save AirspaceAssignment Data

In [ ]:
if do_airspace_assignments:
    print("   started saving airspace_assignments: ", datetime.now())

    df_airspaceassignment_output = (
        df_master
#        .groupBy("track_key")
#        .agg(F.concat_ws(":", F.collect_list("airspace_assignment")).alias("airspace_assignments"))
        .select(
            "track_key",
            "start_date",
            "end_date",
            "orig",
            "dest",
            "callsign",
            "carrier",
            "mode_s_code",
            "aircraft_type",
            "airspace_assignments",
            "airport",
            "operation",
            "month",
        )
    )

    (
        df_airspaceassignment_output
#        .repartition("start_date", "dest", "airport", "operation")
#        .repartition("track_key", "dest", "airport", "operation")
#        .repartition("airport", "operation", "month")
#        .repartition(200)
        .write.option("header", True).partitionBy(["airport", "operation", "month"])
        .csv("CRAFT/" + year0 + "/nas_year_full/airspace_assignments", compression="gzip", mode="append")
    )

    print("   completed saving airspace_assignments: ", datetime.now())